<div style="display: flex; align-items: center; padding: 20px; background-color: #f0f2f6; border-radius: 10px; border: 2px solid #007bff;">
    <img src="../logo.png" style="width: 80px; height: auto; margin-right: 20px;">
    <div style="flex: 1; text-align: left;">
    <h1 style="color: #007bff; margin-bottom: 5px;">GLY 6739.017S26: Computational Seismology</h1>
    <h3 style="color: #666;">Notebook 74:  Local Data, Cloud Data, and Synthetic Data with ObsPy</h3>
    <p style="color: red;"><i>Glenn Thompson | Spring 2026</i></p>
    </div>
</div>



# 1. Local files

# 1.1 Station metadata

We can look up information about many stations in the Federation of Digital Seismic Networks (FDSN) **Station Book**

Information about station II.PFO:
https://www.fdsn.org/station_book/II/PFO/pfo_14.html

You can see that has two seismic sensors:
* An STS-1 broadband seismometer
* An FBA-23 accelerometer

If you click on the **Channel Data** link you will see that it reports channels at several sampling rates:
* 1 Hz
* 20 Hz
* 40 Hz
* 100 Hz (triggered only)

Sensitivities are in Counts/m/s

Note that https://earthquake.usgs.gov/monitoring/operations/stations/II/PFO/ lists other types of sensors. I don't know which is authoritative! But fortunately, we have the actual StationXML file!

Let's load it!


In [ ]:
from obspy import read_inventory
inv = read_inventory("../mess2024/data/station_PFO.xml", format="STATIONXML")
#inv.get_response(st[0].id, st[0].stats.starttime).plot(0.01);

for net in inv:
    for sta in net:
        for cha in sta:
            print(cha)
            cha.response.plot(0.01);

So both the FDSN and USGS sites were half right. Sensor 1 is an STS-1, sensor 2 is a Trillium 240!

# 1.2 Waveform data
Let's read a locally-stored MiniSEED file for station II.PFO:

In [ ]:
from obspy import read, read_inventory
st = read("../mess2024/data/waveform_PFO.mseed")
print(st)
st.plot(equal_scale=False);

The waveform shapes are quite similar, but the amplitudes are different by ~3. Why?

Also, the top trace has a 20 Hz sampling rate, the bottom has a 40 Hz sampling rate. Let's resample the bottom one:

In [ ]:
st[1].resample(sampling_rate=20.0)
print(st)

As you see, there are two Traces, and they have different location codes (00 vs 10). These are from the two different sensors. They are essentially at the same latitude/longitude, but might differ in depth. So we should see the same waveforms right?

Mathematically, we can check that by cross-correlating the signals. To do that, we must make sure they have the same sampling rate - and we see from above that II.PFO.00.BHZ is 20 Hz data, while II.PFO.10.BHZ is 40 Hz data. 



Let's cross correlate the signals from these co-located vertical sensors:

In [ ]:
from obspy.signal.cross_correlation import correlate, xcorr_max
import matplotlib.pyplot as plt

def plot_xcorr(st_in):
    st = st_in.copy()
    fs = min([tr.stats.sampling_rate for tr in st])
    st.resample(sampling_rate=fs)
    cc = correlate(st[0], st[1], int(st[0].stats.sampling_rate))

    plt.figure()
    plt.plot(cc)
    plt.xlabel('Shift (samples)')
    plt.ylabel('Cross correlation coefficient')

    shift, value = xcorr_max(cc)
    print(f'max xcorr is {value} at a shift of {shift} samples')

plot_xcorr(st)

The peak correlation value is 99.27% - very high! But wait, that occurs at a shift of 3 samples. 3 samples at 20 Hz (0.05 s sampling interval) is 0.15 s. Even at a (surface) wave speed of 4 km/s, that suggests the sensors are ~600 m apart!

**What is going on here?**

In [ ]:
st.detrend("linear")
st.taper(max_percentage=0.05, type='cosine')
st.filter("bandpass", freqmin=0.01, freqmax=0.1)
st.plot();

Let's try one more thing:

In [ ]:
st_corrected = st.copy()
st_corrected.remove_response(inventory=inv)
st_corrected.detrend('linear')
st_corrected.plot();
plot_xcorr(st_corrected)

**What changed?**

After removing the instrument response:
* amplitudes are the same
* shift is now 0 samples
* peak correlation coefficient is now 99.75%

### response removal can vary

**water_level** and **pre_filt** are two stabilization tools used during instrument response removal, but they play different roles. 

**water_level** is a regularization parameter applied in the frequency domain during deconvolution. When removing the instrument response, we divide the data spectrum by the response spectrum. At frequencies where the response magnitude becomes very small (for example, outside the flat passband), this division can amplify noise dramatically. The water level sets a minimum floor on the response amplitude—expressed in decibels relative to the peak response—so that the inverse filter cannot exceed a specified amplification factor. It keeps the math stable and prevents catastrophic noise blow-up, but it does not define a physical bandwidth.

**pre_filt**, by contrast, defines the frequency range over which we trust the inversion. It is a four-corner cosine taper (f1, f2, f3, f4) applied in the frequency domain before deconvolution. Between f2 and f3 the inversion is trusted; below f1 and above f4 energy is suppressed; and the regions f1–f2 and f3–f4 are smooth tapers. This is not a “filter” in the usual time-domain sense, but rather a bandwidth constraint that prevents us from trying to recover signal where the instrument response or noise level makes the result unreliable. In short: water level stabilizes the inversion; pre_filt defines the usable frequency band.

We can also use the **output** parameter to select whether to correct to VELocity, ACCeleration, or DISPlacement seismograms. When you integrate (e.g. VEL -> DISP), you blow up low frequencies relative to higher frequencies and add a constant. When you differentiate (e.g. VEL -> ACC) you diminish low frequencies relative to higher frequencies.

In [ ]:
st = read("../mess2024/data/waveform_PFO.mseed")
st.remove_response(inventory=inv, water_level=60, pre_filt=(0.001, 0.002, 8, 10), output="DISP")
st.resample(sampling_rate=20.0)
st.plot();
plot_xcorr(st)


In [ ]:

st = read("../mess2024/data/waveform_PFO.mseed")
st.remove_response(inventory=inv, water_level=None, pre_filt=(0.01, 0.02, 8, 10), output="DISP")
st.resample(sampling_rate=20.0)
st.plot();
plot_xcorr(st)


In [ ]:
st = read("../mess2024/data/waveform_PFO.mseed")
st.remove_response(inventory=inv, water_level=60, pre_filt=None, output="DISP", plot=True)
st.resample(sampling_rate=20.0)
st.plot();
plot_xcorr(st)

Finally, this is how we read events from a local QuakeML file into an ObsPy Catalog object:

In [ ]:
from obspy import read_events

catalog = read_events("../mess2024/data/event_tohoku_with_big_aftershocks.xml")
print(catalog)

# 2. FDSN WS (webservices)

ObsPy has clients to directly fetch data via...

- FDSN webservices (IRIS, Geofon/GFZ, USGS, NCEDC, SeisComp3 instances, ...)
- ArcLink (EIDA, ...)
- Earthworm
- SeedLink (near-realtime servers)
- NERIES/NERA/seismicportal.eu
- NEIC
- SeisHub (local seismological database)

This introduction shows how to use the FDSN webservice client. The FDSN webservice definition is by now the default web service implemented by many data centers world wide. Clients for other protocols work similar to the FDSN client.


## 2.1 Event Data - Tohoku earthquake

Let's find the parameters of the Tohoku earthquake that happened on 2011-03-11

Remember in science, always try to use YYYY-MM-DD date ordering, and absolutely never MM/DD/YYYY. 

And remember to use UTC time. Not local time.

In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

irisclient = Client("IRIS")
t0 = UTCDateTime("2011-03-11")
t1 = t0 + 24 * 3600
cat = irisclient.get_events(starttime=t0, endtime=t1, minmagnitude=9)
print(cat)


Access the event parameters directly

In [ ]:
event = cat[0]
origin = event.preferred_origin()
print(origin, '\n', type(origin))
print(origin.time, '\n', type(origin.time))

print(origin.latitude, origin.longitude)

## 2.2 Station metadata

In [ ]:
# Download station information at the response level!
netcode = "II"
stationcode = "PFO"
inv = irisclient.get_stations(network=netcode, station=stationcode, location="*", channel="BH?",
                     starttime=t0, endtime=t1,
                     level="response")
print(inv)

net = inv.select(network=netcode, station=stationcode)[0]
sta = net.stations[0]
print(sta.latitude, sta.longitude)

### Let's calculate distance from the Tohoku earthquake to station PFO

In [ ]:
from obspy.geodetics import gps2dist_azimuth, locations2degrees

# meters + azimuths
dist_m, az, baz = gps2dist_azimuth(sta.latitude, sta.longitude, origin.latitude, origin.longitude)

# degrees (handy for travel-time models)
dist_deg = locations2degrees(sta.latitude, sta.longitude, origin.latitude, origin.longitude)

print(dist_m/1000, "km")
print(dist_deg, "deg", "\nazimuth", az, "\nback azimuth", baz)

### Let's calculate travel times from the Tohoku earthquake to station PFO


In [ ]:
from obspy.taup import TauPyModel
from obspy.geodetics import locations2degrees
model = TauPyModel(model="iasp91")
arrivals = model.get_travel_times(source_depth_in_km=origin.depth/1000,
                                  distance_in_degree=dist_deg,
                                  phase_list=["P", "S"])
print(arrivals)



More detailed information:

In [ ]:
all_arrivals = model.get_travel_times(source_depth_in_km=origin.depth/1000,
                                  distance_in_degree=dist_deg)

valid_phase_names = [a.name for a in all_arrivals]
print(valid_phase_names)

print(all_arrivals)

**TauP** is a **travel-time calculator** used in seismology to predict when different seismic waves (like P and S waves) will arrive at a station after an earthquake. Given an earthquake’s depth and the distance between the source and a station, TauP computes the expected arrival times of many possible seismic phases based on how waves bend and reflect inside the Earth. It does this using ray theory in a simplified, layered Earth model. In practice, TauP helps us answer questions like: When should the P-wave arrive? What about S, PP, or SKS? These predictions are essential for phase identification, event location, and teaching how seismic waves propagate through the planet.

**IASP91** is one of the **standard Earth models** that TauP can use to make those predictions. It describes how seismic wave speeds change with depth inside the Earth, from the crust through the mantle and into the core. Developed for global seismology, IASP91 assumes the Earth is spherically symmetric and layered, providing average P- and S-wave velocities at different depths. While it’s a simplification of the real Earth, it works remarkably well for many regional and teleseismic problems and has become a widely used reference model. When you run TauP with IASP91, you’re essentially asking: Given this standard picture of Earth’s interior, what travel times should we expect for each seismic phase?

## 2.3 Waveform Data

Now we know that IASP91 model predicts a P-wave will take 713 seconds from the Tohoku earthquake origin to station II.PFO, let's request a waveform timewindow accordingly.

In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

client = Client("IRIS")
print(origin.time, arrivals[0].time, type(origin.time), type(arrivals[0].time))


In [ ]:
ptime = origin.time + arrivals[0].time
stime = origin.time + arrivals[1].time
print("P arrival time:", ptime)
print("S arrival time:", stime)

# compute travel time to PFO
st = client.get_waveforms("II", "PFO", "00", "BH?",
                          ptime - 10 * 60, stime + 30 * 60)
print(st)
st.plot();

Plot with predicted arrival flags added:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

tP = mdates.date2num(ptime.datetime)
tS = mdates.date2num(stime.datetime)

n = len(st)
fig, axes = plt.subplots(n, 1, figsize=(12, 2.2*n), sharex=True)
if n == 1:
    axes = [axes]

for ax, tr in zip(axes, st):
    x = tr.times("matplotlib")  # already absolute matplotlib datenums

    ax.plot(x, tr.data, linewidth=0.8)
    ax.axvline(tP, linestyle="--", color="red", linewidth=1.5, label="P (pred)")
    ax.axvline(tS, linestyle="--", color="green", linewidth=1.5, label="S (pred)")

    ax.set_ylabel(tr.id)
    ax.grid(True, alpha=0.3)

# X-axis formatting
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M:%S"))
axes[-1].xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()

# One legend
axes[0].legend(loc="upper right")
plt.tight_layout()
plt.show()

# 3. Make our own data

## 3.1 Waveform data

In [ ]:
from obspy import Stream, Trace, UTCDateTime
import numpy as np

# --- settings ---
fs = 50.0              # Hz (match Trace.stats.sampling_rate)
n = 1200               # samples (=> 120 s at 50 Hz)
t = np.arange(n) / fs

# Ricker wavelet parameters
t0 = 10.0              # event time (s) within the trace
f0 = 5.0               # dominant frequency (Hz)
Aevent = 1200.0        # event amplitude (bigger than noise+1 Hz sine)

def ricker(t, f0, t0):
    """Ricker (Mexican hat) wavelet centered at t0 with dominant freq f0."""
    tau = t - t0
    a = (np.pi * f0 * tau)
    return (1.0 - 2.0 * a**2) * np.exp(-a**2)

wavelet = Aevent * ricker(t, f0, t0)

st = Stream()
start = UTCDateTime()

# Slightly different event amplitudes per component
event_scales = [1.0, 0.8, 0.6]

for c in range(3):
    noise = 3 * np.random.randint(-100, 100, n)
    signal = 4 * np.sin(2 * np.pi * 1 * t)

    x = noise + signal + event_scales[c] * wavelet

    tr = Trace(data=x.astype(np.float32))
    tr.stats.network = "MV"
    tr.stats.station = f"MB{c:02d}"     # MB00, MB01, MB02
    tr.stats.channel = "BHZ"
    tr.stats.starttime = start
    tr.stats.sampling_rate = fs
    st.append(tr)

print(st)
st.plot();

## 3.2 Event data

In [ ]:
from obspy import UTCDateTime
from obspy.core.event import Catalog, Event, Origin, Magnitude
from obspy.geodetics import FlinnEngdahl

cat = Catalog()
cat.description = "Just a fictitious toy example catalog built from scratch"

e = Event()
e.event_type = "not existing"

o = Origin()
o.time = UTCDateTime(2014, 2, 23, 18, 0, 0)
o.latitude = 47.6
o.longitude = 12.0
o.depth = 10000
o.depth_type = "operator assigned"
o.evaluation_mode = "manual"
o.evaluation_status = "preliminary"
o.region = FlinnEngdahl().get_region(o.longitude, o.latitude)

m = Magnitude()
m.mag = 7.2
m.magnitude_type = "Mw"

m2 = Magnitude()
m2.mag = 7.4
m2.magnitude_type = "Ms"

# also included could be: custom picks, amplitude measurements, station magnitudes,
# focal mechanisms, moment tensors, ...

# make associations, put everything together
cat.append(e)
e.origins = [o]
e.magnitudes = [m, m2]
m.origin_id = o.resource_id
m2.origin_id = o.resource_id

print(cat)
cat.write("/tmp/my_custom_events.xml", format="QUAKEML")
!cat /tmp/my_custom_events.xml

## 3.3 Station metadata

In [ ]:
from obspy.core.inventory import Inventory, Network, Station, Channel, Site, Equipment
from obspy import UTCDateTime

stations = [
    ("MB01", 16.72, -62.54),
    ("MB02", 16.69, -62.51),
    ("MB03", 16.75, -62.48),
    ("MB04", 16.67, -62.47),
    ("MB05", 16.71, -62.56),
    ("MB06", 16.73, -62.50),
]

sensor = Equipment(
    manufacturer="Güralp Systems",
    model="CMG-40T",
    description="Güralp CMG-40T seismometer (teaching example)"
)

datalogger = Equipment(
    manufacturer="Güralp Systems",
    model="Digitizer",
    description="Güralp digitizer (teaching example)"
)

station_objs = []
for code, lat, lon in stations:
    ch = Channel(
        code="HHZ",
        location_code="",
        latitude=lat,
        longitude=lon,
        elevation=500.0,   # placeholder
        depth=0.0,
        azimuth=0.0,
        dip=-90.0,
        sample_rate=100.0,
        sensor=sensor,
        data_logger=datalogger,
    )

    sta = Station(
        code=code,
        latitude=lat,
        longitude=lon,
        elevation=500.0,   # placeholder
        creation_date=UTCDateTime(2020, 1, 1),
        site=Site(name=f"{code} (example)"),
        channels=[ch],
    )
    station_objs.append(sta)

net = Network(
    code="MV",
    stations=station_objs,
    description="Example Montserrat-like teaching network (6 stations, HHZ only)",
)

inv = Inventory(networks=[net], source="Teaching Example")

print(inv)
inv.plot(projection="local")
inv.write("MV_example_inventory.xml", format="STATIONXML")
print("Wrote: MV_example_inventory.xml")